---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 3.2: Add more tools

## 💬 Product Check-in:

>
> **Sam:** "Hey — a few users have mentioned that code examples from the agent don't always run. Syntax errors, missing indentation, that kind of thing. Not a huge volume but it's coming up enough to be worth addressing."
>

**Objective:** Translate next round of feedback into evaluations and scorers that can be used to refine the agent. Refine aspects of the agent to meet updated evaluation criteria.

In this notebook:

---

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()

# Models
MODEL = "gemini-2.5-flash-lite"
JUDGE_MODEL = "gemini:/gemini-3.1-flash-lite-preview"
EXPERIMENT_NAME = os.environ.get("EXPERIMENT_3_NAME", "mlflow-agent-3")
# Model Parameters
TEMPERATURE = 0.2
MAX_TOKENS = 512
# Prompt Versionioning
PROMPT_NAME = "mlflow-agent-system"
PROMPT_ALIAS = f"prompts:/{PROMPT_NAME}@prod"

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.langchain.autolog()

In [3]:
import ast

def validate_python_syntax(code: str) -> dict:
    """
    Validates Python syntax using the built-in ast module.
    No external dependencies required.

    Args:
        code: A string of Python code to validate.

    Returns:
        A dict with keys: valid, error, line, offset
    """
    try:
        ast.parse(code)
        return {
            "valid": True,
            "error": None,
            "line": None,
            "offset": None,
        }
    except SyntaxError as e:
        return {
            "valid": False,
            "error": str(e.msg),
            "line": e.lineno,
            "offset": e.offset,
        }

In [4]:
# Test tool
validate_python_syntax("def foo(x):\n    return x + 1")

{'valid': True, 'error': None, 'line': None, 'offset': None}

In [5]:
validate_python_syntax("public static void main(String[] args) {\n    System.out.println('Hello, world!');\n}")

{'valid': False, 'error': 'invalid syntax', 'line': 1, 'offset': 8}

In [ ]:
# Create Agent
from tool_agent import agent

def predict_fn(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result['messages'][-1].content

In [13]:
predict_fn("What is MLflow?")

["MLflow is an open-source platform designed to manage the end-to-end machine learning lifecycle. It's particularly useful for tracking experiments, packaging code into reproducible runs, and deploying models.\n\nHere are its main components:\n\n*   **MLflow Tracking:** Records parameters, code versions, metrics, and output files when running your ML code. This helps you compare and analyze different training runs.\n*   **MLflow Projects:** Provides a standard format for packaging ML code in a reusable and reproducible way, using code, configuration files, and dependencies.\n*   **MLflow Models:** Offers a convention for packaging ML models that can be used with various downstream tools, along with tools for deploying them.\n*   **MLflow Registry:** A centralized model store, set of APIs, and UI to collaboratively manage the full lifecycle of an MLflow Model, including major and minor versioning, stage transitions, and annotations.\n\nHere's a quick example of how you might log a param

Trace(trace_id=tr-88ccf418f30b11b1d4939b7819d50bdb)

In [18]:
code_quality_evals = [
    {
        # Only one right way to call this — specific args make the expected response tight
        "inputs": {"question": "Show me how to log a metric called 'accuracy' with a value of 0.95 at step 1 in MLflow."},
        "expectations": {
            "expected_response": "mlflow.log_metric('accuracy', 0.95, step=1)",
        },
    },
    {
        # Forces a specific parameter name and value — no ambiguity
        "inputs": {"question": "Show me how to log a parameter called 'learning_rate' with a value of 0.01 in MLflow."},
        "expectations": {
            "expected_response": "mlflow.log_param('learning_rate', 0.01)",
        },
    },
    {
        # Specific enough that the correct answer is a single line
        "inputs": {"question": "Show me how to set an experiment called 'my_experiment' in MLflow."},
        "expectations": {
            "expected_response": "mlflow.set_experiment('my_experiment')",
        },
    },
    {
        # Specific enough that the correct answer is a single line
        "inputs": {"question": "Show me how to search for runs in an experiment called 'my_experiment' in MLflow."},
        "expectations": {
            "expected_response": "mlflow.search_runs(experiment_names=['my_experiment'])",
        },
    },
]
print(f"Code quality eval set size: {len(code_quality_evals)} examples")

Code quality eval set size: 4 examples


# Add examples to full dataset

In [19]:
from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery,
    ToolCallCorrectness,
    ToolCallEfficiency,
    ExpectationsGuidelines
)

results = mlflow.genai.evaluate(
    data=code_quality_evals,
    predict_fn=predict_fn,
    scorers=[
        ToolCallCorrectness(model=JUDGE_MODEL), 
        ToolCallEfficiency(model=JUDGE_MODEL), 
        Correctness(model=JUDGE_MODEL),
        RelevanceToQuery(model=JUDGE_MODEL),
        ExpectationsGuidelines(model=JUDGE_MODEL)
    ],
)

print("=== Evaluation with tool call scorers ===")
print(results.metrics)

2026/04/25 07:26:14 INFO mlflow.genai.scorers.validation: The input data is missing following columns that are required by the specified scorers. The results will be null for those scorers.
 - `guidelines` field in `expectations` column is required by [expectations_guidelines].
2026/04/25 07:26:14 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 4/4 [Elapsed: 00:04, Remaining: 00:00] [predict_fn: 62%, scorers: 38%]
2026/04/25 07:26:21 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'expectations_guidelines': 4/4 failed. Check individual trace assessments for detailed error messages.


=== Evaluation with tool call scorers ===
{'tool_call_efficiency/mean': np.float64(1.0), 'correctness/mean': np.float64(1.0), 'relevance_to_query/mean': np.float64(1.0), 'tool_call_correctness/mean': np.float64(1.0)}


In [20]:
from mlflow.genai.datasets import get_dataset
retrieved_dataset = get_dataset(name="mlflow-agent-eval")

In [22]:
retrieved_dataset.merge_records(code_quality_evals)

In [23]:
from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery,
    ToolCallCorrectness,
    ToolCallEfficiency,
    ExpectationsGuidelines
)

results = mlflow.genai.evaluate(
    data=retrieved_dataset,
    predict_fn=predict_fn,
    scorers=[
        ToolCallCorrectness(model=JUDGE_MODEL), 
        ToolCallEfficiency(model=JUDGE_MODEL), 
        Correctness(model=JUDGE_MODEL),
        RelevanceToQuery(model=JUDGE_MODEL),
        ExpectationsGuidelines(model=JUDGE_MODEL)
    ],
)

print("=== Evaluation with tool call scorers ===")
print(results.metrics)

2026/04/25 08:22:09 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 19/19 [Elapsed: 00:09, Remaining: 00:00] [predict_fn: 61%, scorers: 39%]
2026/04/25 08:22:22 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'expectations_guidelines': 15/19 failed. Check individual trace assessments for detailed error messages.


=== Evaluation with tool call scorers ===
{'correctness/mean': np.float64(0.47368421052631576), 'relevance_to_query/mean': np.float64(1.0), 'tool_call_efficiency/mean': np.float64(0.5263157894736842), 'tool_call_correctness/mean': np.float64(0.7894736842105263), 'expectations_guidelines/mean': np.float64(0.75)}
